# Week 10 · What does the model see? Check a model in five charts

**LDSCI6253 Advanced Information Presentation & Visualisation** · Northeastern University London

This is the lecture's code, to run and change. One dataset throughout: about 30,000 songs and their Spotify audio
features (TidyTuesday, 21 January 2020). At each of the five stops you see the **library's one line** first (seaborn
or scikit-learn's own display), then **build the chart in Altair**, where you choose every colour, order and word.

| § | Stop | The question | The chart | New code |
|---|---|---|---|---|
| 1 | Look first | Is the data fit to learn from? | histogram, box plot | `drop_duplicates`, `bin`, `mark_boxplot` |
| 2 | Too many columns | How do you see ten at once? | brushed matrix, PCA map, k-means groups | `alt.repeat`, `selection_interval`, `PCA`, `KMeans` |
| 3 | What it listens to | Which features matter? | heatmap, importance bars | `.corr()`, `permutation_importance` |
| 4 | Where it's wrong | Is the model any good? | confusion matrix, depth curve, boundary | `confusion_matrix`, `validation_curve` |
| 5 | Tell the public | What does "98%" hide? | the honest comparison | a baseline beside the model |

Run the cells **in order**: each section builds on the one before. Models take a few seconds to train; the depth
curve in §4 takes about half a minute. Needs `altair` (5.4 or later), `pandas`, `scikit-learn`, `seaborn`.

## 0 · Set up

Put `songs.csv` beside this notebook (it is on Canvas); if it is not there, `load` reads it from the course site.
`FEATURES` are the ten audio numbers a model sees; `genre` is one colour per genre, safe for colour-blind readers.

In [ ]:
import altair as alt
import pandas as pd
from pathlib import Path
import numpy as np

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week10/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

songs = load('songs')
FEATURES = ['danceability', 'energy', 'loudness',
            'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo', 'duration_ms']
# Okabe and Ito's colour-blind-safe colours, one per genre
genre = alt.Color('playlist_genre:N', title='genre',
    scale=alt.Scale(
        domain=['edm', 'latin', 'pop', 'r&b', 'rap', 'rock'],
        range=['#0072b2', '#e69f00', '#cc79a7', '#009e73',
               '#d55e00', '#56b4e9']))

# The file is sorted by playlist. Shuffle it once, so any
# slice of it (a sample, a test set) mixes every genre.

## 1a · Look first, the library way

Three lines most notebooks start with: blanks per column, repeated songs, and seaborn's histogram.

In [ ]:
import seaborn as sns

songs.isna().sum()            # missing values, per column
songs.duplicated('track_id').sum()   # the same song twice

sns.histplot(songs, x='track_popularity', bins=50)

## 1b · Look first: what does popularity look like?

One row per song, **shuffled** (the file is sorted by playlist, so any slice of it would be one genre), then a histogram.

In [ ]:
# The file is sorted by playlist. Shuffle it once, so any
# slice of it (a sample, a test set) mixes every genre.
unique = (songs.drop_duplicates('track_id')
          .sample(frac=1, random_state=0))
sample = unique.sample(4000, random_state=0)

alt.Chart(sample).mark_bar().encode(
    x=alt.X('track_popularity:Q', bin=alt.Bin(step=2),
            title='popularity (0 to 100)'),
    y=alt.Y('count()', title='songs'),
).properties(width=520, height=300)

**Try it.** Draw `tempo` instead of `track_popularity` (change the `x` column and its title; try `step=5`). Is there a tower anywhere? (One song has tempo 0.)

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
alt.Chart(sample).mark_bar().encode(
    x=alt.X('tempo:Q', bin=alt.Bin(step=5), title='tempo (beats a minute)'),
    y=alt.Y('count()', title='songs'),
).properties(width=520, height=300)
```
</details>

## 1c · One feature, six genres

A box plot per genre: the middle half of the songs as a box, the median as a line, outliers as dots.

In [ ]:
alt.Chart(sample).mark_boxplot(size=30).encode(
    x=alt.X('playlist_genre:N', title=None),
    y=alt.Y('speechiness:Q', title='speechiness (0 to 1)'),
    color=genre.legend(None),
).properties(width=520, height=300)

**Try it.** Change `speechiness` to `danceability`. Which genre is least danceable?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
alt.Chart(sample).mark_boxplot(size=30).encode(
    x=alt.X('playlist_genre:N', title=None),
    y=alt.Y('danceability:Q'),
    color=genre.legend(None),
).properties(width=520, height=300)
# Rock: its box sits lowest.
```
</details>

## 2a · Ten columns at once, the library way

seaborn's `pairplot`: every pair of columns as a scatter. Quick, but you can't ask it anything.

In [ ]:
import seaborn as sns

few = ['danceability', 'energy', 'speechiness']
sns.pairplot(sample, vars=few, hue='playlist_genre',
             plot_kws={'s': 6})

## 2b · A matrix you can brush

Week 5's brush, on a matrix: drag a box on any panel and the same songs light up in every other panel.

In [ ]:
brush = alt.selection_interval()
small = sample.sample(1500, random_state=1)

alt.Chart(small).mark_point(size=12).encode(
    alt.X(alt.repeat('column'), type='quantitative'),
    alt.Y(alt.repeat('row'), type='quantitative'),
    color=alt.when(brush).then(genre)
             .otherwise(alt.value('#d9dde2')),
).add_params(brush).properties(
    width=125, height=125,
).repeat(
    row=['danceability', 'energy', 'speechiness'],
    column=['danceability', 'energy', 'speechiness'],
)

**Try it.** Add `'valence'` to both lists in `.repeat(...)`. Then drag a box round the songs with the most speechiness: what genre are they?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
# in .repeat(...):
row=['danceability', 'energy', 'speechiness', 'valence'],
column=['danceability', 'energy', 'speechiness', 'valence'],
# The wordiest songs are mostly rap (orange).
```
</details>

## 2c · Ten columns on one map: PCA

Scale every column first, then find the two directions that keep the most variation. The title says how much survives.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(sample[FEATURES])
pca = PCA(n_components=2).fit(scaled)
xy = pca.transform(scaled)
mapped = sample.assign(pc1=xy[:, 0], pc2=xy[:, 1])
kept = pca.explained_variance_ratio_.sum()

alt.Chart(mapped).mark_circle(size=10, opacity=0.6).encode(
    x=alt.X('pc1:Q', title='first direction'),
    y=alt.Y('pc2:Q', title='second direction'),
    color=genre,
    tooltip=['track_name', 'track_artist', 'playlist_genre'],
).properties(width=470, height=330,
    title=f'Two axes keep {kept:.0%} of the variation')

**Try it.** Leave out the scaling: use `sample[FEATURES]` instead of `scaled` in `fit` and `transform`. What happens to the share kept, and why?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
pca = PCA(n_components=2).fit(sample[FEATURES])
xy = pca.transform(sample[FEATURES])
# Almost 100% 'kept': duration_ms (hundreds of thousands) swamps every other column.
# The map is then only song length. Scale first.
```
</details>

## 2d · Let the model find the groups: k-means

**Clustering** is the unsupervised kind of ML: k-means is given the ten numbers and no genres, and finds groups of songs that sit close together. The heatmap shows each group's average on every feature (0 is the average song).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(unique[FEATURES])
groups = KMeans(n_clusters=5, n_init=10,
                random_state=0).fit_predict(scaled)

# What is each group like? Its average on every feature,
# in standard units: 0 is the average song
profile = (pd.DataFrame(scaled, columns=FEATURES)
           .assign(group=[f'group {g + 1}' for g in groups])
           .groupby('group').mean().reset_index()
           .melt('group', var_name='feature', value_name='z'))

alt.Chart(profile).mark_rect().encode(
    x=alt.X('feature:N', sort=FEATURES, title=None),
    y=alt.Y('group:N', title=None),
    color=alt.Color('z:Q', title='above or below average',
        scale=alt.Scale(scheme='redblue', domain=[-1.5, 1.5],
                        reverse=True)),
).properties(width=420, height=210)

**Try it.** Try `n_clusters=3`, then `8`. Then see which genres land in each group: `pd.crosstab(groups, unique['playlist_genre'], normalize='index').round(2)`. Do the groups match the genres?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
groups = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(scaled)
pd.crosstab(groups, unique['playlist_genre'], normalize='index').round(2)
# The groups cut across genres: they are closer to moods (wordy, calm, energetic).
```
</details>

## 3a · Which columns move together?

Correlation between every pair of columns, as a heatmap with its numbers on it. Correlation runs from −1 to 1, so the colour scale diverges from white at 0.

In [ ]:
corr = (unique[FEATURES].corr().reset_index()
        .melt('index', var_name='other', value_name='r'))

base = alt.Chart(corr).encode(
    x=alt.X('index:N', title=None, sort=FEATURES),
    y=alt.Y('other:N', title=None, sort=FEATURES))
cells = base.mark_rect().encode(
    color=alt.Color('r:Q', scale=alt.Scale(
        scheme='redblue', domain=[-1, 1], reverse=True)))
words = base.mark_text(fontSize=9).encode(
    text=alt.Text('r:Q', format='.1f'))

(cells + words).properties(width=380, height=380)

**Try it.** Change the scheme to `'blues'` and remove `domain=[-1, 1]`. Why is that worse?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
color=alt.Color('r:Q', scale=alt.Scale(scheme='blues'))
# −0.5 and +0.5 now look like 'a bit' and 'a lot': the sign is lost.
# A diverging scale with 0 in the middle shows both directions.
```
</details>

## 3b · What does the model listen to?

The first model: three hundred decision trees that vote, trained on three songs in four. Then shuffle one column at a time on the songs it never saw: how much accuracy is lost?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X, y = unique[FEATURES], unique['playlist_genre']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
model = RandomForestClassifier(300, random_state=0)
model.fit(X_tr, y_tr)        # learn from 3 songs in 4

from sklearn.inspection import permutation_importance
shuffled = permutation_importance(
    model, X_te, y_te, n_repeats=5, random_state=0)
scores = pd.DataFrame({'feature': FEATURES,
    'lost': shuffled.importances_mean})

alt.Chart(scores).mark_bar(color='#0c3354').encode(
    x=alt.X('lost:Q', axis=alt.Axis(format='%'),
            title='accuracy lost when the feature is shuffled'),
    y=alt.Y('feature:N', sort='-x', title=None),
).properties(width=460, height=300)

**Try it.** Draw the forest's own count instead: `pd.DataFrame({'feature': FEATURES, 'lost': model.feature_importances_})`. Which feature comes first now?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
scores = pd.DataFrame({'feature': FEATURES,
    'lost': model.feature_importances_})
# then the same alt.Chart(scores)... cell.
# The two rankings differ: the forest's own count favours columns with many different values.
```
</details>

## 3c · Two ways to rank the same features

The forest's own count (made while it trained) beside the shuffle (measured on test songs it never saw), so you can see them disagree.

In [ ]:
both = pd.DataFrame({'feature': FEATURES,
    'the forest’s own count': model.feature_importances_,
    'shuffle and see': shuffled.importances_mean,
}).melt('feature', var_name='method', value_name='importance')

alt.Chart(both).mark_bar(color='#0c3354').encode(
    x=alt.X('importance:Q', title=None),
    y=alt.Y('feature:N', sort='-x', title=None),
).properties(width=200, height=260).facet(
    column=alt.Column('method:N', title=None),
).resolve_scale(x='independent', y='independent')

**Try it.** Which feature moves most between the two rankings? Why might a column with many different values (like `duration_ms`) score higher in the forest's own count?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
# The forest's own count favours columns with many different values:
# a tree can split them in many places, so they get used more often.
# The shuffle asks a fairer question: how much worse is the model, on new songs, without it?
```
</details>

## 4a · Score it: right, compared with what, for whom?

**Accuracy** is the share of test songs the model gets right. The **baseline** is the score of a rule that learns nothing: always the commonest genre. **Recall** for one genre is the share of its real songs the model gets right.

In [ ]:
guess = model.predict(X_te)            # songs it never saw
accuracy = (guess == y_te).mean()         # share it got right
baseline = y_te.value_counts(normalize=True).max()
labels = sorted(y.unique())
found = pd.DataFrame({'genre': labels, 'recall': [
    (guess[y_te == g] == g).mean() for g in labels]})

bars = alt.Chart(found).mark_bar().encode(
    x=alt.X('recall:Q', axis=alt.Axis(format='%'),
            scale=alt.Scale(domain=[0, 1]),
            title='share of each genre’s songs it got right'),
    y=alt.Y('genre:N', sort='-x', title=None),
    color=genre.legend(None))
lines = alt.Chart(pd.DataFrame({'at': [accuracy, baseline],
    'what': ['all songs', 'always the commonest genre']})
).mark_rule(strokeDash=[5, 3]).encode(x='at:Q')
names = lines.mark_text(align='left', dx=4, y=-8).encode(
    text='what:N')
(bars + lines + names).properties(width=420, height=230)

**Try it.** Print the three numbers: `print(f'{accuracy:.0%} right, baseline {baseline:.0%}')` and `found.sort_values('recall')`. Which genre has the lowest recall?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
print(f'{accuracy:.0%} right, baseline {baseline:.0%}')
found.sort_values('recall')
# Latin and pop are lowest: their playlists overlap.
```
</details>

## 4b · Where is it wrong? The library way

scikit-learn draws the confusion matrix straight from the model. Read a row: of the real songs of that genre, what did the model call them?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    model, X_te, y_te, normalize='true',
    values_format='.0%', cmap='Blues')

## 4c · A confusion matrix in Altair

The same matrix, built: you choose the colours, the order and the words. Each row adds up to 100%.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(y.unique())
counts = confusion_matrix(y_te, model.predict(X_te),
                          labels=labels)
share = counts / counts.sum(axis=1, keepdims=True)
cm = pd.DataFrame(share, index=labels, columns=labels)
cm = (cm.reset_index(names='true')
        .melt('true', var_name='guessed', value_name='share'))

base = alt.Chart(cm).encode(
    x=alt.X('guessed:N', title='the model guessed'),
    y=alt.Y('true:N', title='the real genre'))
cells = base.mark_rect().encode(
    color=alt.Color('share:Q', scale=alt.Scale(scheme='blues'),
                    legend=None))
words = base.mark_text().encode(
    text=alt.Text('share:Q', format='.0%'),
    color=alt.when(alt.datum.share > 0.5)
             .then(alt.value('white')).otherwise(alt.value('black')))

(cells + words).properties(width=360, height=360)

**Try it.** Show counts instead of shares: use `counts` in place of `share` when making `cm`, and `format='d'` for the text. Why are shares fairer for comparing genres?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
cm = pd.DataFrame(counts, index=labels, columns=labels)
# ... and text=alt.Text('share:Q', format='d'), with domain off the colour scale
# Genres have different numbers of songs; shares put every row on the same 0–100% footing.
```
</details>

## 4d · Is it learning, or memorising?

One tree, allowed to grow deeper and deeper. Grey: how it scores on the songs it learnt from. Red: on songs it never saw.

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

depths = list(range(1, 26))
train, test = validation_curve(
    DecisionTreeClassifier(random_state=0), X, y,
    param_name='max_depth', param_range=depths, cv=3)
curve = pd.DataFrame({'depth': depths,
    'songs it learnt from': train.mean(axis=1),
    'songs it never saw': test.mean(axis=1),
}).melt('depth', var_name='scored on', value_name='accuracy')

alt.Chart(curve).mark_line(point=True).encode(
    x=alt.X('depth:Q', title='how deep the tree may grow'),
    y=alt.Y('accuracy:Q', axis=alt.Axis(format='%')),
    color=alt.Color('scored on:N', scale=alt.Scale(
        range=['#9aa3ad', '#c8102e'])),
).properties(width=470, height=300)

**Try it.** Find the best depth in code: `curve[curve['scored on'] == 'songs it never saw'].sort_values('accuracy').tail(1)`. Mark it on the chart with a rule.

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
best = curve[curve['scored on'] == 'songs it never saw'].nlargest(1, 'accuracy')
rule = alt.Chart(best).mark_rule(strokeDash=[4, 3]).encode(x='depth:Q')
(alt.Chart(curve).mark_line(point=True).encode(
    x='depth:Q', y=alt.Y('accuracy:Q', axis=alt.Axis(format='%')),
    color='scored on:N') + rule).properties(width=470, height=300)
```
</details>

## 4e · Where does the model draw the line?

Ask the tree about a grid of made-up songs, and colour each point of the grid by its answer: that is the boundary. Altair never sees the model, only a table of its answers.

In [ ]:
three = unique[unique['playlist_genre']
                .isin(['pop', 'rap', 'rock'])]
two = ['speechiness', 'danceability']
depth = 3
tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
tree.fit(three[two], three['playlist_genre'])

# Ask the tree about every point of a grid: its answer is the boundary
# 60 x 60 = 3,600 points: under Altair's 5,000-row limit
gx, gy = np.meshgrid(np.linspace(0, 0.5, 60), np.linspace(0.2, 1, 60))
grid = pd.DataFrame({'speechiness': gx.ravel(),
                     'danceability': gy.ravel()})
grid['playlist_genre'] = tree.predict(grid[two])

x = alt.X('speechiness:Q', scale=alt.Scale(domain=[0, 0.5]),
          axis=alt.Axis(values=[0, 0.1, 0.2, 0.3, 0.4, 0.5]))
y = alt.Y('danceability:Q', scale=alt.Scale(domain=[0.2, 1]))
area = alt.Chart(grid).mark_square(size=55, opacity=0.3).encode(
    x=x, y=y, color=genre)
seen = three.sample(600, random_state=0).query(
    'speechiness <= 0.5 and danceability >= 0.2')
dots = alt.Chart(seen).mark_circle(size=16, stroke='white',
    strokeWidth=0.5).encode(x=x, y=y, color=genre)

(area + dots).properties(width=400, height=330)

**Try it.** Change `depth = 3` to `depth = 20` and run it again. Would this boundary work on next year's songs?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
depth = 20
# ... the rest of the cell unchanged.
# The boundary wraps round single songs: it has memorised them. New songs won't sit in the same tiny islands.
```
</details>

## 4f · The boundary, the library way

scikit-learn's `DecisionBoundaryDisplay` does the grid for you. Its default margin (`eps=1.0`) is meant for unscaled data: try deleting `eps=0.02` to see why it is set.

In [ ]:
import numpy as np
from sklearn.inspection import DecisionBoundaryDisplay

shown = DecisionBoundaryDisplay.from_estimator(
    tree, three[two], response_method='predict',
    alpha=0.4, eps=0.02, cmap='tab10')
few = three.sample(600, random_state=0)
shown.ax_.scatter(few['speechiness'], few['danceability'],
    c=few['playlist_genre'].astype('category').cat.codes,
    cmap='tab10', vmax=9, s=8, edgecolor='white', linewidth=0.3)

## 5 · Tell the public: the honest comparison

A "hit" is popularity 80 or more: 1.8% of songs. The model is right 98.1% of the time; saying "not a hit" every time is right 98.2%. The chart puts the three numbers a reader needs side by side.

In [ ]:
hit = unique['track_popularity'] >= 80     # a "hit"
X_tr, X_te, h_tr, h_te = train_test_split(
    X, hit, test_size=0.25, random_state=0, stratify=hit)
spotter = RandomForestClassifier(300, random_state=0)
guess = spotter.fit(X_tr, h_tr).predict(X_te)

scores = pd.DataFrame({'claim': [
    'the hit model is right',
    'saying "not a hit" every time is right',
    'real hits the model found'],
  'share': [(guess == h_te).mean(), 1 - h_te.mean(),
            guess[h_te.to_numpy()].mean()]})

alt.Chart(scores).mark_bar(color='#0c3354').encode(
    x=alt.X('share:Q', axis=alt.Axis(format='%'),
            scale=alt.Scale(domain=[0, 1]), title=None),
    y=alt.Y('claim:N', sort=None, title=None),
).properties(width=420, height=180) + alt.Chart(scores).mark_text(
    align='left', dx=6).encode(
    x='share:Q', y=alt.Y('claim:N', sort=None),
    text=alt.Text('share:Q', format='.1%'))

**Try it.** Lower the bar for a hit to 70 and run it again. Does the model find more of the real hits? Is "98% accurate" still the right headline?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
hit = unique['track_popularity'] >= 70
# ... the rest unchanged. More songs count as hits, the lazy model's score falls,
# and the share of real hits found is still the number a reader needs.
```
</details>

## Next: the lab

Open `10_Activity_IPDV_Model_Check.ipynb`. A new question for the same five checks: **can the audio tell R&B from
rap?** You will look first, find which columns differ, train a model, find where it is wrong, and write the honest
headline.